This notebook covers the advanced S3 features that architects use for data management at scale: replication across regions and accounts, lifecycle policies that automatically tier and expire data, event notifications, S3 Select for in-place querying, and Object Lock for immutable storage.

## S3 Replication

S3 replication automatically copies objects from a **source bucket** to a **destination bucket** — asynchronously, after each upload.

### Types

| Type | Buckets | Use case |
|---|---|---|
| **CRR** (Cross-Region Replication) | Different regions | DR, compliance (data residency), lower latency for global users |
| **SRR** (Same-Region Replication) | Same region, different accounts or buckets | Log aggregation, dev/test copies, compliance within region |

### Requirements
- **Versioning must be enabled** on both source and destination buckets
- An IAM role must grant S3 permission to replicate objects on your behalf
- For cross-account replication, the destination bucket policy must allow the source account's replication role

### What is replicated
- New objects uploaded after replication is configured
- Object metadata, ACLs (if configured), object tags
- Encrypted objects (SSE-S3 and SSE-KMS — SSE-KMS requires extra configuration to share the key)

### What is NOT replicated by default
- Objects uploaded **before** replication was configured (use S3 Batch Replication for existing objects)
- Delete markers (can be optionally enabled)
- Objects in Glacier or Glacier Deep Archive
- Objects replicated from another bucket (no chaining — A→B does not trigger B→C)

### Delete marker replication
By default, if you delete an object in the source bucket, the delete marker is **not** replicated. This protects against accidental deletion cascading to the destination. You can optionally enable delete marker replication.

### Replication Time Control (RTC)
Standard replication is best-effort. **RTC** provides an SLA: 99.99% of new objects replicated within **15 minutes**. RTC also includes replication metrics in CloudWatch. Use it when you need a guaranteed RPO for DR.

## S3 Lifecycle Policies

Lifecycle rules automate transitioning objects between storage classes and expiring (deleting) objects — eliminating manual cost management.

### Rule scope
Each rule applies to:
- The entire bucket, or
- A specific **prefix** (e.g. `logs/`), or
- Objects with specific **tags**, or
- Objects within a **size range** (min/max bytes)

### Transition actions
Move objects to a cheaper storage class after N days from creation (or last modified).

```
Day 0     → S3 Standard        (upload)
Day 30    → S3 Standard-IA     (transition)
Day 90    → S3 Glacier IR      (transition)
Day 365   → S3 Glacier Deep Archive  (transition)
Day 2555  → Delete             (expiration)
```

### Valid transition paths

```
Standard → Standard-IA → One Zone-IA
         → Intelligent-Tiering
         → Glacier IR → Glacier Flexible → Glacier Deep Archive
```

You can only move **down** (to cheaper classes). You cannot transition back up via lifecycle.

### Minimum days between transitions
- Standard → Standard-IA or One Zone-IA: **minimum 30 days**
- Standard-IA → Glacier IR: **minimum 30 days** (90 days total from Standard)

### Expiration actions
- **Delete current version** after N days (non-versioned or versioned buckets)
- **Expire non-current versions** after N days (versioned buckets)
- **Delete expired delete markers** — clean up delete markers with no non-current versions beneath them
- **Abort incomplete multipart uploads** after N days — prevents orphaned parts accumulating storage charges

### Lifecycle for versioned buckets
```
Rule 1: Transition current versions to Standard-IA after 30 days
Rule 2: Expire non-current versions after 60 days    ← saves cost
Rule 3: Delete expired delete markers                ← housekeeping
Rule 4: Abort incomplete multipart uploads after 7 days
```

## S3 Event Notifications

S3 can emit events when objects are created, deleted, restored, or replicated — triggering downstream processing.

### Event types
- `s3:ObjectCreated:*` — PUT, POST, COPY, multipart complete
- `s3:ObjectRemoved:*` — delete, delete marker created
- `s3:ObjectRestore:*` — Glacier restore initiated/completed
- `s3:Replication:*` — replication failure
- `s3:LifecycleTransition`, `s3:IntelligentTiering`

### Destinations

| Destination | Use case |
|---|---|
| **SQS** | Queue events for reliable async processing |
| **SNS** | Fan out to multiple subscribers |
| **Lambda** | Immediate processing (resize image, extract metadata, validate) |
| **EventBridge** | Route to many targets with filtering, retry, and archiving |

### EventBridge vs direct notifications
EventBridge receives **all** S3 events (when enabled) and offers advanced filtering, multiple targets per rule, event archiving, and replay. For complex routing or multiple downstream targets, prefer EventBridge over direct SQS/SNS/Lambda notifications.

```
S3 upload
  │
  ├── Direct: Lambda → resize image
  │
  └── EventBridge:
        ├── Rule 1 (*.jpg) → Lambda (thumbnail)
        ├── Rule 2 (*.csv) → SQS (ETL pipeline)
        └── Rule 3 (any)   → CloudWatch Logs (audit)
```

## S3 Object Lock

Object Lock implements **WORM** (Write Once, Read Many) storage — objects cannot be modified or deleted for a specified retention period.

### Retention modes

| Mode | Who can override | Use case |
|---|---|---|
| **Governance** | Users with special IAM permission (`s3:BypassGovernanceRetention`) | Internal protection with an override escape hatch |
| **Compliance** | Nobody — not even root | Strict regulatory compliance (SEC, FINRA, HIPAA) |

### Retention period
A fixed number of days or until a specific date. After the period expires, the object can be deleted normally.

### Legal Hold
An indefinite lock with no expiry date — used during litigation or investigation. Any user with `s3:PutObjectLegalHold` permission can place or remove a legal hold, independent of any retention period.

### Requirements
- Object Lock must be enabled when the bucket is created (cannot be added later)
- Versioning is automatically enabled
- A default retention mode and period can be set at the bucket level

> **Exam tip:** Compliance mode = no one can delete, not even root. Governance mode = protected, but override is possible with the right permission.

## S3 Select & Glacier Select

**S3 Select** lets you run SQL queries against the content of an S3 object — filtering and projecting data server-side without downloading the entire file.

Supported formats: CSV, JSON, Parquet. Compression: GZIP, BZIP2.

```python
# Query only the rows you need from a large CSV
response = s3.select_object_content(
    Bucket='my-bucket',
    Key='data/sales-2024.csv',
    ExpressionType='SQL',
    Expression="SELECT * FROM S3Object WHERE country = 'US' AND amount > 1000",
    InputSerialization={'CSV': {'FileHeaderInfo': 'USE'}},
    OutputSerialization={'JSON': {}}
)
```

**Glacier Select** does the same for objects in Glacier Flexible Retrieval — filter at the archive level before paying for a full restore.

Both reduce data transfer and processing costs significantly compared to downloading the entire object.

## S3 Batch Operations

**S3 Batch Operations** runs a single action across billions of objects in one managed job.

| Operation | Description |
|---|---|
| **Copy** | Copy objects to another bucket (or same bucket with different metadata) |
| **Invoke Lambda** | Run a Lambda function on every object |
| **Replace ACL** | Set ACL on all objects |
| **Put Object Tagging** | Add/replace tags on all objects |
| **Delete Object Tagging** | Remove tags |
| **Restore from Glacier** | Initiate Glacier restore on all matching objects |
| **Replicate** | Backfill existing objects into a replication destination |

Batch Operations generates a **completion report** and sends notifications. You provide an inventory CSV or S3 Inventory file as the input manifest.

Common use case: you set up replication today, but need to also copy the millions of objects that existed before replication was configured. Run a Batch Replication job.

## S3 Storage Lens

**S3 Storage Lens** provides organisation-wide visibility into S3 usage, activity, and cost optimisation opportunities.

- Works across all accounts in an AWS Organization
- Surfaced in a single dashboard with 29+ usage and activity metrics
- Highlights: unused buckets, objects without lifecycle rules, non-current version storage, incomplete multipart uploads
- Recommendations for cost savings (e.g. "these buckets have no lifecycle rules and 80% of data is > 90 days old")
- Metrics can be exported to S3 for querying with Athena

## Working with Replication, Lifecycle & Events via boto3

In [ ]:
import boto3
import json

s3 = boto3.client('s3', region_name='us-east-1')

In [ ]:
# Configure CRR from us-east-1 to eu-west-1
s3.put_bucket_replication(
    Bucket='source-bucket',
    ReplicationConfiguration={
        'Role': 'arn:aws:iam::123456789012:role/s3-replication-role',
        'Rules': [
            {
                'ID': 'replicate-all',
                'Status': 'Enabled',
                'Filter': {},          # empty = replicate everything
                'Destination': {
                    'Bucket': 'arn:aws:s3:::destination-bucket-eu',
                    'StorageClass': 'STANDARD_IA',
                    'ReplicationTime': {
                        'Status': 'Enabled',
                        'Time': {'Minutes': 15}  # RTC — 15 min SLA
                    },
                    'Metrics': {'Status': 'Enabled', 'EventThreshold': {'Minutes': 15}}
                },
                'DeleteMarkerReplication': {'Status': 'Disabled'}
            }
        ]
    }
)
print("CRR configured with RTC")

In [ ]:
# Lifecycle policy: tier logs, expire old versions, abort incomplete uploads
s3.put_bucket_lifecycle_configuration(
    Bucket='source-bucket',
    LifecycleConfiguration={
        'Rules': [
            {
                'ID': 'tier-logs',
                'Status': 'Enabled',
                'Filter': {'Prefix': 'logs/'},
                'Transitions': [
                    {'Days': 30,  'StorageClass': 'STANDARD_IA'},
                    {'Days': 90,  'StorageClass': 'GLACIER'},
                    {'Days': 365, 'StorageClass': 'DEEP_ARCHIVE'},
                ],
                'Expiration': {'Days': 2555}   # delete after 7 years
            },
            {
                'ID': 'expire-old-versions',
                'Status': 'Enabled',
                'Filter': {},
                'NoncurrentVersionExpiration': {'NoncurrentDays': 60},
                'ExpiredObjectDeleteMarker': True
            },
            {
                'ID': 'abort-multipart',
                'Status': 'Enabled',
                'Filter': {},
                'AbortIncompleteMultipartUpload': {'DaysAfterInitiation': 7}
            }
        ]
    }
)
print("Lifecycle policy applied")

In [ ]:
# Configure S3 event notification → Lambda on any object creation
s3.put_bucket_notification_configuration(
    Bucket='source-bucket',
    NotificationConfiguration={
        'LambdaFunctionConfigurations': [
            {
                'Id': 'process-uploads',
                'LambdaFunctionArn': 'arn:aws:lambda:us-east-1:123456789012:function:process-upload',
                'Events': ['s3:ObjectCreated:*'],
                'Filter': {
                    'Key': {
                        'FilterRules': [
                            {'Name': 'prefix', 'Value': 'uploads/'},
                            {'Name': 'suffix', 'Value': '.jpg'}
                        ]
                    }
                }
            }
        ]
    }
)
print("Event notification configured")

In [ ]:
# Create a bucket with Object Lock (must be at creation time)
s3.create_bucket(
    Bucket='compliance-archive-bucket',
    ObjectLockEnabledForBucket=True
)

# Set default retention: Compliance mode, 2555 days (7 years)
s3.put_object_lock_configuration(
    Bucket='compliance-archive-bucket',
    ObjectLockConfiguration={
        'ObjectLockEnabled': 'Enabled',
        'Rule': {
            'DefaultRetention': {
                'Mode': 'COMPLIANCE',
                'Days': 2555
            }
        }
    }
)
print("Compliance bucket with 7-year retention created")

In [ ]:
# S3 Select — query a CSV file without downloading it
response = s3.select_object_content(
    Bucket='source-bucket',
    Key='data/sales.csv',
    ExpressionType='SQL',
    Expression="SELECT s.order_id, s.amount FROM S3Object s WHERE s.country = 'US' AND CAST(s.amount AS FLOAT) > 500",
    InputSerialization={
        'CSV': {'FileHeaderInfo': 'USE', 'RecordDelimiter': '\n', 'FieldDelimiter': ','}
    },
    OutputSerialization={'JSON': {'RecordDelimiter': '\n'}}
)

for event in response['Payload']:
    if 'Records' in event:
        print(event['Records']['Payload'].decode('utf-8'), end='')
    elif 'Stats' in event:
        stats = event['Stats']['Details']
        print(f"\nScanned: {stats['BytesScanned']:,} bytes  "
              f"Returned: {stats['BytesReturned']:,} bytes")

## Summary

| Concept | Key Takeaway |
|---|---|
| CRR | Cross-region replication — DR, compliance, latency; versioning required on both buckets |
| SRR | Same-region replication — log aggregation, cross-account copies |
| Replication scope | Only new objects after config; existing objects need Batch Replication |
| Delete marker replication | Disabled by default — protects destination from accidental deletes |
| RTC | 15-minute replication SLA with CloudWatch metrics |
| Lifecycle transitions | Automate tiering; minimum 30 days Standard → IA |
| Lifecycle expiration | Delete current versions, expire non-current, abort incomplete multipart |
| Event notifications | Trigger SQS/SNS/Lambda/EventBridge on object events |
| EventBridge | Preferred for complex routing — multiple targets, filtering, replay |
| Object Lock — Governance | Protected, but override possible with special IAM permission |
| Object Lock — Compliance | Immutable — nobody can delete, not even root |
| Legal Hold | Indefinite lock; no expiry; independent of retention period |
| S3 Select | SQL queries on S3 objects server-side — reduce data transfer and cost |
| Batch Operations | Run one action across billions of objects; includes Batch Replication |
| Storage Lens | Org-wide S3 analytics and cost optimisation recommendations |